# 20 · NULLs & Three-Valued Logic

`NULL` means **unknown**, and it makes SQL logic *three-valued*: expressions can
be `TRUE`, `FALSE`, or `UNKNOWN`. Mishandling NULLs is one of the most common
sources of subtle SQL bugs. This module makes you bulletproof.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## `NULL` is not equal to anything — not even `NULL`
`= NULL` is never true; it evaluates to `UNKNOWN`, and `WHERE` only keeps `TRUE`
rows. Always test with `IS NULL` / `IS NOT NULL`.

In [ ]:
%%sql
SELECT
    (NULL = NULL)     AS null_eq_null,     -- NULL (unknown), not 1
    (NULL <> 1)       AS null_ne_one,      -- NULL
    (NULL IS NULL)    AS null_is_null;     -- 1 (true)

Rows with a missing email — only `IS NULL` finds them:

In [ ]:
%%sql
SELECT customer_id, first_name, email FROM customers WHERE email IS NULL;

## ⚠️ The `NOT IN` + NULL trap
If the list/subquery behind `NOT IN` contains **even one NULL**, the whole
predicate becomes `UNKNOWN` and you get **zero rows** — a notorious bug. Watch:

In [ ]:
%%sql
WITH ids(id) AS (VALUES (1), (2), (NULL))
SELECT customer_id, first_name
FROM customers
WHERE customer_id NOT IN (SELECT id FROM ids);   -- returns NOTHING!

The robust fix is `NOT EXISTS`, which is NULL-safe:

In [ ]:
%%sql
WITH ids(id) AS (VALUES (1), (2), (NULL))
SELECT customer_id, first_name
FROM customers cu
WHERE NOT EXISTS (SELECT 1 FROM ids WHERE ids.id = cu.customer_id)
ORDER BY customer_id;

## NULLs in aggregates, `DISTINCT`, and `GROUP BY`
- Aggregates **ignore** NULLs: `AVG`/`SUM`/`COUNT(col)` skip them (so `AVG` can
  differ from `SUM/COUNT(*)`).
- `GROUP BY` treats all NULLs as **one** group.
- `DISTINCT` treats NULLs as **equal** (collapses to one).

In [ ]:
%%sql
SELECT COUNT(*)      AS total_rows,
       COUNT(email)  AS non_null_emails,   -- ignores the NULL
       COUNT(DISTINCT country) AS distinct_countries
FROM customers;

## Taming NULLs: `COALESCE`, `IFNULL`, `NULLIF`
- `COALESCE(a, b, c)` → first non-NULL
- `IFNULL(a, b)` → two-argument shortcut
- `NULLIF(a, b)` → NULL when `a = b` (great to avoid divide-by-zero)

In [ ]:
%%sql
SELECT first_name,
       COALESCE(email, 'no-email@unknown') AS email,
       IFNULL(email, 'none')               AS email_short
FROM customers
LIMIT 6;

`NULLIF` guarding a division (returns NULL instead of erroring when the denominator is 0):

In [ ]:
%%sql
SELECT 100.0 / NULLIF(0, 0) AS safe_divide;

## Sorting NULLs
By default SQLite sorts NULLs **first** in ascending order. Use
`NULLS LAST` / `NULLS FIRST` to control it explicitly:

In [ ]:
%%sql
SELECT first_name, email FROM customers ORDER BY email NULLS LAST LIMIT 6;

## Practice

**✏️ Exercise 1.** Count how many customers are missing an email, using a NULL-aware condition.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT COUNT(*) AS missing_email FROM customers WHERE email IS NULL;

**✏️ Exercise 2.** Show each customer's email, substituting the text 'N/A' when it is missing.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT first_name, COALESCE(email, 'N/A') AS email FROM customers;

### ✅ Recap
`NULL` = unknown → three-valued logic. Compare with `IS [NOT] NULL`, avoid the
`NOT IN`+NULL trap (use `NOT EXISTS`), remember aggregates skip NULLs while
`GROUP BY`/`DISTINCT` fold them together, and reach for
`COALESCE`/`IFNULL`/`NULLIF` to handle them.

**Next:** `21_data_modeling_and_normalization.ipynb`.